# Pipeline 3: Donation Amount Prediction

**Organization:** River of Life / Lighthouse Sanctuary (INTEX)  
**Methodology:** CRISP-DM–aligned (see `pipeline_guide.md` in this folder)

---

## Executive Summary

We estimate how large a supporter’s **next monetary gift** is likely to be, using only giving history available **before** a snapshot date—so fundraising can set **ethical ask bands** and forecast cash.

**What this notebook delivers**
- Grouped validation by donor (no cheating with duplicate rows)
- Baseline + Ridge (explain) + Random Forest (predict)
- Full deployment: PostgreSQL + .NET fundraising UI

*Non-technical readers:* skim the Executive Summary, **Business Interpretation**, **Key Findings**, and **Recommended Actions**, then use charts in Sections 3–5 for discussion with data staff.

---


## 1. Problem Framing

### Business problem
Campaign copy and CRM defaults often use one-size ask strings. Knowing **expected next gift size** (with uncertainty) helps personalize asks without embarrassing low-ball or exploitative high-ball amounts.

### Stakeholders
| **Development / comms** | Ask strings, email tiering |
| **Finance / ED** | Rough cash timing (with caveats) |
| **Donors** | Respectful, proportional invitations |

### Why this matters
Better alignment between **capacity signals in data** and **outreach** can lift response without increasing pressure tactics.

### Predictive goal (what we forecast or score)
**Continuous:** predicted **peso amount** of the **first** monetary gift in the 120-day forward window (logged target during training).

### Explanatory goal (what we want to understand)
**Directional drivers:** which RFM-style factors historically align with **larger vs smaller** next gifts (Ridge coefficients on log scale).

### Why predictive and explanatory are different
Prediction optimizes **RMSE** for operational scoring; explanation optimizes **human trust** and workshop teaching—even if a linear model is slightly worse on error.

### Decision this work supports
**Ask-band assignment** in CRM and **campaign segment budgets** (e.g., major-gift officer prep for top decile).

### Limitations (preview)
Past wealth and channel confound **causal** stories; see Section 6.

---


## Data Validity & Leakage Check

### How the target is defined
**y** = `amount` of the **earliest** monetary donation with `as_of_date` < `donation_date` ≤ `as_of_date + 120 days`.

### What information is allowed at prediction time
Only gifts with `donation_date` < `as_of_date`, plus static supporter attributes. Allocation diversity uses rows with `donation_date` < `as_of_date`.

### Why future information does not leak into features
The forward gift that defines **y** is never included in feature aggregation windows.

### Why the train/test approach is valid
**GroupShuffleSplit** and **GroupKFold** by `supporter_id` because monthly snapshots duplicate donors.

### Automated checks in this notebook
Code prints row counts and date ranges; inspect `panel` for missing labels.

---


## 2. Data Acquisition & Preparation

**Tables:** `donations.csv` (Monetary, non-null `amount`), `supporters.csv`, `donation_allocations.csv`.

**Joins:** `donations` ⋈ `supporters` on `supporter_id`; `donations` ⋈ `allocations` on `donation_id` for `n_program_areas`.

**Engineering:** winsorize historical amount sums at 1%/99%; target `log1p(y_amount)` for training.

---


In [ ]:
import json
import warnings
from datetime import timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns  # noqa: F401 — used in EDA cells across generated notebooks

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk", font_scale=0.85)

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "lighthouse_csv_v7").is_dir():
            return p
    raise FileNotFoundError(
        "Could not find lighthouse_csv_v7. Open or run from the INTEX II EDA project folder."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "lighthouse_csv_v7"
OUTPUT_DIR = PROJECT_ROOT / "ml_pipelines" / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def winsorize(s, lo=0.01, hi=0.99):
    return s.clip(lower=s.quantile(lo), upper=s.quantile(hi))

sup = pd.read_csv(DATA_DIR / "supporters.csv", parse_dates=["created_at", "first_donation_date"])
don = pd.read_csv(DATA_DIR / "donations.csv", parse_dates=["donation_date"])
alloc = pd.read_csv(DATA_DIR / "donation_allocations.csv")
don_a = don.merge(alloc, on="donation_id", how="left")
mon = don[(don["donation_type"] == "Monetary") & don["amount"].notna()].copy()

def build_panel(horizon=120):
    rows = []
    anchors = pd.date_range("2023-04-01", "2025-10-01", freq="MS")
    mx = mon["donation_date"].max()
    for as_of in anchors:
        if as_of + timedelta(days=horizon) > mx:
            continue
        for sid in mon.loc[mon["donation_date"] < as_of, "supporter_id"].unique():
            pm = mon[(mon["supporter_id"] == sid) & (mon["donation_date"] < as_of)].sort_values("donation_date")
            if pm.empty:
                continue
            fut = mon[(mon["supporter_id"] == sid) & (mon["donation_date"] > as_of) & (mon["donation_date"] <= as_of + timedelta(days=horizon))]
            if fut.empty:
                continue
            y_amt = float(fut.iloc[0]["amount"])
            last = pm.iloc[-1]
            recency = (as_of - last["donation_date"]).days
            freq = len(pm)
            amt_sum = winsorize(pm["amount"]).sum()
            amt_mean = pm["amount"].mean()
            has_rec = int(pm["is_recurring"].fillna(False).astype(bool).any())
            tenure = (as_of - pm["donation_date"].min()).days
            da = don_a[(don_a["supporter_id"] == sid) & (don_a["donation_date"] < as_of)]
            n_prog = int(da["program_area"].nunique(dropna=True))
            rows.append({"supporter_id": sid, "as_of_date": as_of, "y_amount": y_amt, "recency_days": recency, "frequency": freq, "sum_amount_hist": amt_sum, "mean_amount_hist": amt_mean, "has_recurring": has_rec, "tenure_days": tenure, "n_program_areas": n_prog})
    df = pd.merge(pd.DataFrame(rows), sup[["supporter_id", "supporter_type", "region", "acquisition_channel"]], on="supporter_id")
    return df

panel = build_panel()
panel["log_y"] = np.log1p(panel["y_amount"])
print(panel.shape, panel["y_amount"].describe())
panel.head()


## 3. Exploration (EDA)

We visualize distributions and relationships **relevant to the business question**, not generic plots. Narrative interpretation follows each chart in markdown where noted.

---


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(panel["y_amount"], kde=True, ax=ax[0])
ax[0].set_title("Next gift (PHP)")
sns.histplot(panel["log_y"], kde=True, ax=ax[1])
ax[1].set_title("log1p(next gift)")
plt.tight_layout()
plt.show()
sns.heatmap(panel[["recency_days", "frequency", "sum_amount_hist", "mean_amount_hist", "tenure_days", "y_amount"]].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Linear associations (not causal)")
plt.tight_layout()
plt.show()


## 4. Modeling & Feature Selection

### Feature rationale (why these inputs)
Classic **RFM** plus **recurring** flag and **program breadth** (`n_program_areas`) capture habit, scale, and engagement depth.

### Three-model strategy
1. **Baseline — DummyRegressor (predict training-set mean of log1p y):** trivial rule so we never mistake “model” for “signal.”
2. **Interpretable — Ridge regression (scaled numerics + one-hot):** coefficients or clear structure for **explanation** and stakeholder trust.
3. **Performance — Random Forest regressor (+ Gradient Boosting comparison in CV):** stronger fit for **batch scoring**; may sacrifice some interpretability.

### Feature selection
We keep the feature set **parsimonious** and justified; where helpful, regularization (Ridge / L1) or tree-based implicit selection reduces noise. Final model choice is documented in Section 5 with **tradeoffs**.

---


In [ ]:
NUM = ["recency_days", "frequency", "sum_amount_hist", "mean_amount_hist", "has_recurring", "tenure_days", "n_program_areas"]
CAT = ["supporter_type", "region", "acquisition_channel"]
X = panel[NUM + CAT]
y = panel["log_y"]
groups = panel["supporter_id"]
prep = ColumnTransformer([("n", StandardScaler(), NUM), ("c", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])

def pipe(model):
    return Pipeline([("p", prep), ("m", model)])

gkf = GroupKFold(5)
rows_cv = []
for name, model in [
    ("baseline_mean", DummyRegressor(strategy="mean")),
    ("ridge", Ridge(alpha=2.0)),
    ("rf", RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=4, random_state=RANDOM_STATE, n_jobs=-1)),
    ("gbrt", GradientBoostingRegressor(random_state=RANDOM_STATE, max_depth=3, n_estimators=150, learning_rate=0.06)),
]:
    sc = cross_validate(pipe(model), X, y, cv=gkf, groups=groups, scoring={"rmse": "neg_root_mean_squared_error", "r2": "r2"}, n_jobs=-1)
    rows_cv.append({"model": name, "rmse_log": -sc["test_rmse"].mean(), "r2_log": sc["test_r2"].mean()})
cv_df = pd.DataFrame(rows_cv)
print(cv_df.to_string(index=False))

gss = GroupShuffleSplit(1, test_size=0.25, random_state=RANDOM_STATE)
tr, te = next(gss.split(X, y, groups))
best = RandomForestRegressor(n_estimators=250, max_depth=6, min_samples_leaf=4, random_state=RANDOM_STATE, n_jobs=-1)
p_best = pipe(best)
p_best.fit(X.iloc[tr], y.iloc[tr])
pred_log = p_best.predict(X.iloc[te])
true_amt = np.expm1(y.iloc[te])
pred_amt = np.expm1(pred_log)
print("Holdout MAE (PHP)", mean_absolute_error(true_amt, pred_amt))
print("Holdout RMSE (PHP)", mean_squared_error(true_amt, pred_amt) ** 0.5)
print("Holdout R2 log-scale", r2_score(y.iloc[te], pred_log))


In [ ]:
ridge_full = pipe(Ridge(2.0))
ridge_full.fit(X.iloc[tr], y.iloc[tr])
coef = pd.Series(ridge_full.named_steps["m"].coef_, index=ridge_full.named_steps["p"].get_feature_names_out()).sort_values()
plt.figure(figsize=(8, max(4, len(coef) * 0.15)))
coef.plot(kind="barh", color="steelblue")
plt.title("Ridge coefficients (log1p next amount)")
plt.tight_layout()
plt.show()
print("Chosen for production scoring: Random Forest (see CV table vs Ridge vs baseline).")


## 5. Evaluation & Interpretation

### Metrics
We report metrics appropriate to the task (regression: MAE, RMSE, R²; classification: accuracy, precision, recall, F1, ROC-AUC where applicable). **Grouped or held-out units** (donor, resident, safehouse) avoid optimistic scores when the same entity appears many times.

### What to look for
- **Lift over baseline:** if the interpretable and performance models barely beat the baseline, treat outputs as **weak decision support** until more data arrives.
- **Stability:** cross-validation spread indicates whether the model generalizes or chases noise.

---


## Business Interpretation

### What this means in plain English
The model learns **who tends to give larger next gifts** from past frequency and scale. It does **not** read donor morality—only observable history.

### How reliable is this for real decisions?
With ~60 unique donors and many panel rows, metrics **move** with split. Use scores as **priorities**, not guarantees; refresh quarterly.

### What should the organization do differently?
Set **ask bands** by decile of predicted next gift; **A/B test** wording; never show raw scores to donors.

### What decision does this directly support?
**CRM default ask** and **staff prep** for high-decile donors.

### When the model is wrong
- **False positives (predicted high risk / high amount / etc., but reality was “fine”):**  
  Suggested ask **too high** → lower conversion or donor discomfort; mitigate with **soft ranges** and human override.
- **False negatives (model said “low concern” but something important happened):**  
  Suggested ask **too low** → leave money on table; mitigate by **caps and major-gift review** for high-tenure donors.

---


## 6. Causal & Relationship Analysis

### What relationships showed up in the data
Larger historical totals and frequency usually align with larger next gifts in CV.

### Why these are not proven causal
Channel and acquisition confound wealth; Ridge weights are **associational**.

### Honest limitations
No experimental variation; small donor universe.

---


## Key Findings

- Tree model typically beats Ridge on RMSE; Ridge explains **direction** of drivers.
- Program breadth and recurring signal often matter—validate on your next refresh.
- Always compare to **baseline mean** before trusting production scores.

---


## Recommended Actions

- Deploy **decile bands** in CRM, not point asks alone.
- Log `model_version` and retrain when campaigns shift behavior.
- Pair scores with **relationship manager judgment** for major gifts.

---


## 7. Deployment Plan

### What triggers scoring
**Nightly or weekly batch** after donations sync (or on-demand before big campaigns).

### Where results appear in the .NET application
**Fundraising → Donor detail** panel: “Suggested next gift band”; optional internal-only column in donor list export.

### Who uses the output and how
| **Development** | Sets asks |
| **Comms** | Email tiering |
| **Data admin** | Monitors drift |

### PostgreSQL table schema

```sql
CREATE TABLE donor_next_amount_predictions (
  prediction_id BIGSERIAL PRIMARY KEY,
  supporter_id INTEGER NOT NULL,
  as_of_date DATE NOT NULL,
  forward_window_days INTEGER NOT NULL DEFAULT 120,
  predicted_next_amount_php DOUBLE PRECISION NOT NULL,
  band_low_php DOUBLE PRECISION,
  band_high_php DOUBLE PRECISION,
  model_version VARCHAR(40) NOT NULL,
  scored_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
  UNIQUE (supporter_id, as_of_date, forward_window_days, model_version)
);
CREATE INDEX idx_donor_amt_sup ON donor_next_amount_predictions (supporter_id);
CREATE INDEX idx_donor_amt_run ON donor_next_amount_predictions (scored_at DESC);
```

### Example upsert (batch job after training/scoring)

```sql
INSERT INTO donor_next_amount_predictions
  (supporter_id, as_of_date, forward_window_days, predicted_next_amount_php, band_low_php, band_high_php, model_version)
VALUES (42, DATE '2026-04-07', 120, 1250.0, 800.0, 1700.0, 'donation_amount_v2')
ON CONFLICT (supporter_id, as_of_date, forward_window_days, model_version)
DO UPDATE SET predicted_next_amount_php = EXCLUDED.predicted_next_amount_php,
  band_low_php = EXCLUDED.band_low_php, band_high_php = EXCLUDED.band_high_php, scored_at = NOW();
```

### Python → PostgreSQL → .NET data flow
1. Scheduled **batch job** (e.g., nightly Airflow / Azure Function / Windows Task Scheduler) runs this notebook’s scoring script or a `joblib` loader.
2. Script reads the latest warehouse export or DB replica, builds features **as of `run_date`**, computes predictions.
3. Results are **UPSERTed** into the table below (idempotent per natural key).
4. The **.NET** admin API reads via EF Core or Dapper; UI shows sortable lists, filters, and **no raw model internals** to end users unless “explain” panel is explicitly designed.

Persist `joblib.dump(final_pipeline, OUTPUT_DIR / 'donation_amount_v2.joblib')` after sign-off; scoring script loads artifact and uses SQLAlchemy/`psycopg2` `execute_values` for bulk upsert.

---


In [ ]:
from joblib import dump
final = Pipeline([("p", prep), ("m", RandomForestRegressor(250, max_depth=6, min_samples_leaf=4, random_state=RANDOM_STATE, n_jobs=-1))])
final.fit(X, y)
dump(final, OUTPUT_DIR / "donation_amount_v2.joblib")
print("Saved", OUTPUT_DIR / "donation_amount_v2.joblib")
